In [ ]:
import pandas as pd
import numpy as np

#loading dataset
df = pd.read_csv('indian_heart_health_survey_10k.csv')

# a)Standardizing Column Names
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

# b)Checking and remove=ing duplicates
initial_count = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {initial_count - len(df)} duplicate rows.")

#c) Quick Inspection
print(df.info())

Removed 0 duplicate rows.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   age                           10000 non-null  int64 
 1   gender                        10000 non-null  object
 2   location_tier                 10000 non-null  object
 3   city_name                     10000 non-null  object
 4   living_area_type              10000 non-null  object
 5   air_quality_level             10000 non-null  object
 6   aqi_device_available          10000 non-null  object
 7   education_level               8010 non-null   object
 8   occupation_type               10000 non-null  object
 9   family_size                   10000 non-null  int64 
 10  family_income_per_year        10000 non-null  object
 11  smoker_in_house               10000 non-null  object
 12  fuel_type_used                10000 non-null  obj

In [ ]:
# 1)Fill missing education_level based on the most frequent (mode) education for that occupation
# much more accurate than filling with a static 'Unknown'
df['education_level'] = df.groupby('occupation_type')['education_level'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Not Disclosed")
)

# 2) Verifing Check if nulls are now 0
print(f"Missing values after imputation: {df['education_level'].isnull().sum()}")

Missing values after imputation: 0


In [ ]:
# 1)income Ranking (Ordinal Encoding)
# Converting text like '< 3L' into numbers (1, 2, 3, 4) for mathematical analysis
income_map = {'< 3L': 1, '3–6L': 2, '6–10L': 3, '10L+': 4}
df['income_rank'] = df['family_income_per_year'].map(income_map)

# 2. Creating the 'High_Risk_Flag' (Composite Feature)
# We define 'High Risk' (1) if a person
# has at least 2 out of 4 clinical "Red Flags".
def define_risk(row):
    red_flags = 0
    if row['known_hypertension'] == 'Yes': red_flags += 1
    if row['known_diabetes'] == 'Yes': red_flags += 1
    if row['heart_symptoms_past_year'] == 'Yes': red_flags += 1
    if row['family_history_heart_disease'] == 'Yes': red_flags += 1
    return 1 if red_flags >= 2 else 0

df['high_risk_flag'] = df.apply(define_risk, axis=1)

# 3) Age Binning (Categorical )
# Helps the Power BI team create better visuals for different generations
df['age_group'] = pd.cut(df['age'], bins=[0, 30, 50, 75, 120], labels=['Young', 'Adult', 'Mid-Senior', 'Senior'])

print("Feature Engineering Complete. Created: income_rank, high_risk_flag, age_group")

Feature Engineering Complete. Created: income_rank, high_risk_flag, age_group


In [ ]:
# 1)Text Normalization: Enforce Title Case & Strip Spaces
# This ensures "mumbai" and " Mumbai " both become "Mumbai" for SQL Joins
text_columns = df.select_dtypes(include=['object']).columns
for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.title()

# 2)Outlier Detection (The Audit)
# We check if Age is within human limits (0-110) and Family Size is > 0
age_outliers = df[(df['age'] < 0) | (df['age'] > 110)]
family_outliers = df[df['family_size'] < 1]

print(f"Audit Findings - Age Outliers: {len(age_outliers)}")
print(f"Audit Findings - Family Size Outliers: {len(family_outliers)}")

# If outliers found, this line removes them to ensure statistical integrity
df = df[(df['age'] >= 0) & (df['age'] <= 110)]
df = df[df['family_size'] >= 1]

print("Data Consistency & Audit Complete + all text normalized and outliers verified.")

Audit Findings - Age Outliers: 0
Audit Findings - Family Size Outliers: 0
Data Consistency & Audit Complete + all text normalized and outliers verified.


In [ ]:
# 1)Droping administrative/unnecessary columns that aren't needed for analysis
# This makes the file lighter and easier for my SQL import
cols_to_drop = ['consent_for_use', 'data_sharing_consent']
final_df = df.drop(columns=cols_to_drop)

# 2)Exportingg the final dataset
final_df.to_csv('heart_health_FINAL_CLEANED.csv', index=False)


from google.colab import files
files.download('heart_health_FINAL_CLEANED.csv')

print("Date cleaning done ")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Date cleaning done 
